# StressLoad(t)

Este notebook implementa el modelado supervisado de `StressLoad(t)` a partir del dataset maestro generado en `1_02_WESAD_preprocesamiento.ipynb`, y se organiza en cuatro bloques principales.

El **bloque 1** prepara los datos para el entrenamiento: se reservan cuatro sujetos como conjunto de validación final (nunca usados en el desarrollo del modelo), se define la variable objetivo y se construyen dos configuraciones de variables de entrada — **Camino A** (todas las señales disponibles: HR, HRV, EDA, TEMP) y **Camino B** (únicamente HR y sus derivadas temporales) —, esta última pensada para ser transferible a wearables comerciales de consumo que no exponen EDA ni TEMP.

El **bloque 2** entrena y evalúa un conjunto de modelos mediante validación cruzada Leave-One-Subject-Out (LOSO) sobre ambos caminos, con desglose de resultados por sujeto para identificar heterogeneidad en el rendimiento.

El **bloque 3** optimiza los modelos mediante búsqueda de hiperparámetros con CV anidada, evalúa el efecto del balanceo de clases y cierra con una comparación cuantitativa entre Camino A y Camino B.

Los **bloques 4 y 5** son estudios de ablación que verifican empíricamente dos decisiones de diseño tomadas previamente por criterio conceptual: si la normalización por baseline aporta mejora real de rendimiento sobre HR bruto, y si la inclusión de acelerometría (ACC) mejora Camino A. Ambos estudios concluyen manteniendo la decisión original, ahora con respaldo empírico y no solo conceptual.

El notebook cierra con el entrenamiento del modelo final sobre Camino B y una nota de mejora futura pendiente (test estadístico formal para la comparación Camino A vs. Camino B).

# 0. Imports

Se importan las librerías necesarias para el modelado: `scikit-learn` para los clasificadores y la validación cruzada, `xgboost` como extensión al benchmark original de Schmidt et al. (2018), y las utilidades estándar de manejo de datos (`pandas`, `numpy`, `pathlib`). Se fija una semilla (`SEED = 42`) para garantizar la reproducibilidad de los modelos estocásticos (Random Forest, AdaBoost, XGBoost); los modelos deterministas (Decision Tree, LDA, kNN) no requieren semilla.

In [1]:
# ======================================================================================================
# 0. INSTALACIÓN DE ELEMENTOS NECESARIOS (descomenta si hace falta en tu entorno)
# ======================================================================================================
# !pip install xgboost

# ======================================================================================================
# 1. IMPORTS
# ======================================================================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd

# --- Selección de modelo / validación ---
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV

# --- Clasificadores (réplica del benchmark de Schmidt et al. 2018 + XGBoost) ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
import xgboost as xgb

# --- Métricas ---
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Semilla fija, para reproducibilidad (afecta a RF, AdaBoost, XGBoost — no a DT/LDA/kNN, que son deterministas)
SEED = 42

# 1. Carga de los datos

Se carga el dataset maestro a resolución de 1 minuto generado por `WESAD_preprocesamiento.ipynb`, leyendo la ruta desde el mismo fichero de configuración externo (`WESAD_config_rutas.json`) que usa el notebook de preprocesamiento. Se carga el conjunto completo de columnas; la selección de variables para Camino A y Camino B se realiza más adelante, dentro de este mismo notebook.

In [2]:
# ======================================================================================================
# 2. CARGA DEL DATASET MAESTRO (1 MINUTO) GENERADO POR EL NOTEBOOK DE PREPROCESAMIENTO
# ======================================================================================================

# Cargamos el archivo de configuración de rutas (mismo que usa el notebook de preprocesamiento)
ruta_config = Path("WESAD_config_rutas.json")
if not ruta_config.exists():
    raise FileNotFoundError("⚠️ No se encontró 'WESAD_config_rutas.json'. Asegúrate de que esté en la misma carpeta que este notebook.")

with open(ruta_config, "r", encoding="utf-8") as f:
    config = json.load(f)

# Apuntamos específicamente al parquet de 1 MINUTO (no al de 1Hz) — es el que tiene
# los estadísticos agregados, rolling y lags que construimos en el notebook de preprocesamiento
ruta_parquet = Path(config["rutas_procesamiento"]["ruta_salida_parquet_1min"])
if not ruta_parquet.exists():
    raise FileNotFoundError(f"⚠️ No se encontró el parquet en {ruta_parquet}. ¿Has vuelto a ejecutar el notebook de preprocesamiento tras el cambio de lags?")

# Cargamos el dataset completo (todas las columnas — Camino A y Camino B se filtran después, en este notebook)
df = pd.read_parquet(ruta_parquet, engine="fastparquet")

# Control de calidad: dimensiones, sujetos presentes y listado de columnas,
# para verificar visualmente que las columnas nuevas de hoy (lags de EDA/TEMP/HRV) están ahí
print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"   Sujetos presentes: {sorted(df['ID_sujeto'].unique())}")
print(f"\n📋 Columnas disponibles ({df.shape[1]}):")
for col in df.columns:
    print(f"   {col}")

✅ Dataset cargado: 1425 filas x 89 columnas
   Sujetos presentes: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

📋 Columnas disponibles (89):
   ID_sujeto
   minuto_id
   HR_mean
   HRV_RMSSD_mean
   HRV_SDNN_mean
   ACC_mean
   EDA_mean
   TEMP_mean
   Estado_Emocional
   Etiqueta
   ventana_pura
   HR_mean_roll_mean_3m
   HR_mean_roll_std_3m
   HRV_RMSSD_mean_roll_mean_3m
   HRV_RMSSD_mean_roll_std_3m
   EDA_mean_roll_mean_3m
   EDA_mean_roll_std_3m
   TEMP_mean_roll_mean_3m
   TEMP_mean_roll_std_3m
   ACC_mean_roll_mean_3m
   ACC_mean_roll_std_3m
   HR_mean_roll_mean_5m
   HR_mean_roll_std_5m
   HRV_RMSSD_mean_roll_mean_5m
   HRV_RMSSD_mean_roll_std_5m
   EDA_mean_roll_mean_5m
   EDA_mean_roll_std_5m
   TEMP_mean_roll_mean_5m
   TEMP_mean_roll_std_5m
   ACC_mean_roll_mean_5m
   ACC_mean_roll_std_5m
   HR_mean_roll_mean_10m
   HR_mean_roll_std_10m
   HRV_RMSSD_mean_roll_mean_10m
   HRV_RMSSD_mean_roll_std_10m
   EDA_mean_roll_mean_1

## 1.1. Selección de cuatro sujetos aleatorios para validacion final

Antes de cualquier entrenamiento, se reservan aleatoriamente 4 de los 15 sujetos de WESAD (`seed=42`) como conjunto totalmente independiente, destinado a los escenarios sintéticos de validación posteriores y nunca utilizado en el desarrollo ni la optimización del modelo. Los 11 sujetos restantes constituyen el conjunto de entrenamiento sobre el que se aplica validación cruzada Leave-One-Subject-Out (LOSO). Se comprueba que la partición no genera déficit de la clase minoritaria (Estresado) en ninguno de los dos grupos.

In [3]:
import random

SEED = 42
random.seed(SEED)

ALL_SUBJECTS = sorted(df['ID_sujeto'].unique().tolist())
assert len(ALL_SUBJECTS) == 15, f"Se esperaban 15 sujetos, hay {len(ALL_SUBJECTS)}"

RESERVED_SUBJECTS = sorted(random.sample(ALL_SUBJECTS, 4))
LOSO_SUBJECTS = sorted([s for s in ALL_SUBJECTS if s not in RESERVED_SUBJECTS])

print(f"Reservados para escenarios sintéticos (n=4): {RESERVED_SUBJECTS}")
print(f"LOSO / entrenamiento (n=11): {LOSO_SUBJECTS}\n")

# --- Comprobación de sensatez: minutos de 'Estresado' por sujeto ---
minutos_por_sujeto = (
    df.groupby(['ID_sujeto', 'Estado_Emocional'])
    .size()
    .unstack(fill_value=0)
)

print("Reservados — minutos por clase:")
print(minutos_por_sujeto.loc[RESERVED_SUBJECTS])
print(f"Total 'Estresado' en reservados: {minutos_por_sujeto.loc[RESERVED_SUBJECTS, 'Estresado'].sum()}\n")

print("LOSO — minutos por clase:")
print(minutos_por_sujeto.loc[LOSO_SUBJECTS])
print(f"Total 'Estresado' en LOSO: {minutos_por_sujeto.loc[LOSO_SUBJECTS, 'Estresado'].sum()}")

min_estresado_loso = minutos_por_sujeto.loc[LOSO_SUBJECTS, 'Estresado'].min()
print(f"\nMínimo de 'Estresado' en un sujeto de LOSO: {min_estresado_loso}")
if min_estresado_loso == 0:
    print("ALERTA: al menos un sujeto de LOSO se queda sin minutos 'Estresado' — hay que discutirlo antes de fijar la partición.")

Reservados para escenarios sintéticos (n=4): ['S10', 'S11', 'S5', 'S6']
LOSO / entrenamiento (n=11): ['S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S7', 'S8', 'S9']

Reservados — minutos por clase:
Estado_Emocional  Diversion  Estresado  Meditacion  Otros  Relajado  \
ID_sujeto                                                             
S10                       6         12          14      3        19   
S11                       7         11          13      2        20   
S5                        6         11          12      3        20   
S6                        6         11          14      2        19   

Estado_Emocional  Transicion  
ID_sujeto                     
S10                       37  
S11                       33  
S5                        44  
S6                        63  
Total 'Estresado' en reservados: 45

LOSO — minutos por clase:
Estado_Emocional  Diversion  Estresado  Meditacion  Otros  Relajado  \
ID_sujeto                                      

La homogeneidad de minutos "Estresado" entre grupos no es un efecto del sorteo, sino un artefacto del propio protocolo WESAD: el bloque de estrés (TSST) tiene una duración estandarizada (~10-12 minutos) para todos los sujetos por diseño experimental, por lo que la varianza entre sujetos en esta clase es baja de partida. La partición aleatoria (`seed=42`) resulta en una distribución sana de la clase minoritaria en ambos grupos —reservados: 11-12 minutos por sujeto; LOSO: rango equivalente—, sin ningún caso degenerado. Esta comprobación confirma que la partición no introduce déficit de clase positiva en ningún fold de LOSO, consecuencia de la duración estandarizada del bloque de estrés en el protocolo WESAD y no de un resultado favorable del muestreo aleatorio.

## 1.2. Definición del target y filtrado de sujetos

Se define el problema como clasificación binaria, replicando el diseño de Schmidt et al. (2018): Estresado (etiqueta 2) frente a no-estrés (Relajado + Diversión, etiquetas 1 y 3), excluyendo Transición, Meditación y Otros. Se excluyen además los 4 sujetos reservados para los escenarios sintéticos, quedando un dataset de entrenamiento de 398 filas sobre 11 sujetos, con una distribución de clases de 278 (no-estrés) frente a 120 (Estresado).

In [4]:
RESERVED_SUBJECTS = ['S5', 'S6', 'S10', 'S11']

# Filtro binario: Estresado (Etiqueta=2) vs. Relajado+Diversión (1,3)
# Fuera: Transición (0), Meditación (4), Otros (5-7) — réplica exacta de Schmidt et al. (2018)
etiquetas_binario = [1, 2, 3]
df_bin = df[df['Etiqueta'].isin(etiquetas_binario)].copy()

# Target binario: 1 = Estresado, 0 = no-estrés (Relajado/Diversión)
df_bin['target'] = (df_bin['Etiqueta'] == 2).astype(int)

# Excluye los 4 sujetos reservados para escenarios sintéticos — no entran en el entrenamiento/LOSO del clasificador
df_loso = df_bin[~df_bin['ID_sujeto'].isin(RESERVED_SUBJECTS)].copy()

print(f"Filas tras filtro binario (los 15 sujetos): {df_bin.shape[0]}")
print(f"Filas tras excluir reservados (dataset LOSO): {df_loso.shape[0]}")
print(f"\nDistribución de clases en df_loso:")
print(df_loso['target'].value_counts())
print(f"\nSujetos en df_loso ({df_loso['ID_sujeto'].nunique()}): {sorted(df_loso['ID_sujeto'].unique())}")
assert df_loso['ID_sujeto'].nunique() == 11, "Deberían quedar 11 sujetos para LOSO"

Filas tras filtro binario (los 15 sujetos): 545
Filas tras excluir reservados (dataset LOSO): 398

Distribución de clases en df_loso:
target
0    278
1    120
Name: count, dtype: int64

Sujetos en df_loso (11): ['S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S7', 'S8', 'S9']


El dataset de entrenamiento resultante presenta un desbalance moderado de clases (278 no-estrés frente a 120 Estresado, ratio ≈2,3:1), consistente con la naturaleza del protocolo experimental: el bloque de estrés (TSST) es más corto que la suma de las condiciones de reposo y diversión. Este desbalance se tiene en cuenta más adelante al evaluar el efecto del balanceo de clases sobre el rendimiento del modelo (Sección 3.3).

## 1.3. Creación de datasets Camino A y Camino B


Se excluyen las columnas de rolling y lag de 3 minutos del conjunto de entrenamiento (Camino A y B), aunque el dataset maestro las incluya.

**Motivo de diseño:** un rolling de 3 minutos se sitúa entre el lag (1-2 min, dinámica inmediata) y el rolling ya cerrado de 5 minutos (tendencia de corto-medio plazo) — no cubre una escala temporal genuinamente distinta, sino que solapa con ambas. Con lecturas consecutivas de HR a 1 minuto de resolución, la autocorrelación entre ventanas de 1-2 min y 3-5 min es alta; añadir la ventana de 3 minutos aporta poca información nueva y sí más dimensionalidad sobre un dataset de solo 398 filas y 11 sujetos.

Esta decisión se valida empíricamente mediante ablación en la Sección 6 (tras contar con los resultados base de Camino A y Camino B).

In [5]:
columnas_metadata = ['ID_sujeto', 'minuto_id', 'Etiqueta', 'Estado_Emocional', 'ventana_pura', 'target']

# Camino A: HR + EDA + TEMP + HRV (solo medias normalizadas y sus derivadas temporales) — sin ACC
prefijos_camino_A = ('HR_', 'HRV_', 'EDA_', 'TEMP_')
# Camino B: solo HR (mismo tratamiento temporal que Camino A)
prefijos_camino_B = ('HR_',)

# Se excluyen las ventanas de 3 minutos (rolling y lag) — probadas y descartadas empíricamente,
# ver justificación en el apartado: no aportan mejora y solapan con lag 1-2m / rolling 5m ya cerrados.
columnas_camino_A = [
    c for c in df_loso.columns
    if c.startswith(prefijos_camino_A) and '_norm' in c and '_3m_' not in c and not c.endswith('_3m')
]
columnas_camino_B = [
    c for c in df_loso.columns
    if c.startswith(prefijos_camino_B) and '_norm' in c and '_3m_' not in c and not c.endswith('_3m')
]

print(f"Camino A (transferible, sin ventanas de 3m): {len(columnas_camino_A)} columnas")
for c in columnas_camino_A:
    print(f"  {c}")

print(f"\nCamino B (transferible, sin ventanas de 3m): {len(columnas_camino_B)} columnas")
for c in columnas_camino_B:
    print(f"  {c}")

# Comprobaciones de sensatez
assert not any(c.startswith('ACC') for c in columnas_camino_A), "ACC no debería estar en Camino A"
assert not any(c.startswith('ACC') for c in columnas_camino_B), "ACC no debería estar en Camino B"
assert all('_norm' in c for c in columnas_camino_A), "Solo deberían quedar columnas normalizadas"
assert all('_norm' in c for c in columnas_camino_B), "Solo deberían quedar columnas normalizadas"
assert not any('_3m' in c for c in columnas_camino_A), "Se ha colado una columna de 3 minutos"
assert not any('_3m' in c for c in columnas_camino_B), "Se ha colado una columna de 3 minutos"
assert set(columnas_camino_B).issubset(set(columnas_camino_A)), "Camino B debería ser subconjunto de Camino A"
assert not set(columnas_camino_A) & set(columnas_metadata), "No debería haber solapamiento con metadata"

Camino A (transferible, sin ventanas de 3m): 29 columnas
  HR_mean_norm
  HRV_RMSSD_mean_norm
  HRV_SDNN_mean_norm
  EDA_mean_norm
  TEMP_mean_norm
  HR_mean_roll_mean_5m_norm
  HR_mean_roll_std_5m_norm
  HRV_RMSSD_mean_roll_mean_5m_norm
  HRV_RMSSD_mean_roll_std_5m_norm
  EDA_mean_roll_mean_5m_norm
  EDA_mean_roll_std_5m_norm
  TEMP_mean_roll_mean_5m_norm
  TEMP_mean_roll_std_5m_norm
  HR_mean_roll_mean_10m_norm
  HR_mean_roll_std_10m_norm
  HRV_RMSSD_mean_roll_mean_10m_norm
  HRV_RMSSD_mean_roll_std_10m_norm
  EDA_mean_roll_mean_10m_norm
  EDA_mean_roll_std_10m_norm
  TEMP_mean_roll_mean_10m_norm
  TEMP_mean_roll_std_10m_norm
  HR_mean_norm_lag_1m_norm
  HR_mean_norm_lag_2m_norm
  HRV_RMSSD_mean_norm_lag_1m_norm
  HRV_RMSSD_mean_norm_lag_2m_norm
  EDA_mean_norm_lag_1m_norm
  EDA_mean_norm_lag_2m_norm
  TEMP_mean_norm_lag_1m_norm
  TEMP_mean_norm_lag_2m_norm

Camino B (transferible, sin ventanas de 3m): 7 columnas
  HR_mean_norm
  HR_mean_roll_mean_5m_norm
  HR_mean_roll_std_5m_norm
 

Camino A queda constituido por 29 columnas (HR, HRV, EDA y TEMP, con sus versiones instantáneas, rolling de 5 y 10 minutos, y lags de 1 y 2 minutos), mientras que Camino B se reduce a las 7 columnas equivalentes derivadas únicamente de HR. Las comprobaciones de sensatez confirman que Camino B es subconjunto estricto de Camino A, que ninguna columna de ACC ni de ventana de 3 minutos se ha colado en ninguno de los dos caminos, y que no hay solapamiento con las columnas de metadata.

## 1.4. Comprobación final de Nans

Antes de entrenar cualquier modelo, se verifica la ausencia de valores nulos en las columnas de Camino A (superconjunto de Camino B). El dataset de 398 filas no presenta ningún NaN, confirmando que la imputación aplicada en el preprocesamiento (`WESAD_preprocesamiento.ipynb`) y los descartes de columnas realizados hasta este punto no han introducido huecos en los datos de entrenamiento.

In [6]:
nan_por_columna = df_loso[columnas_camino_A].isna().sum()
columnas_con_nan = nan_por_columna[nan_por_columna > 0]

print("Columnas de Camino A con NaN (y cuántos):")
print(columnas_con_nan if not columnas_con_nan.empty else "Ninguna — no hay NaN en Camino A")

print(f"\nFilas totales en df_loso: {df_loso.shape[0]}")
print(f"Filas con al menos un NaN en Camino A: {df_loso[columnas_camino_A].isna().any(axis=1).sum()}")

Columnas de Camino A con NaN (y cuántos):
Ninguna — no hay NaN en Camino A

Filas totales en df_loso: 398
Filas con al menos un NaN en Camino A: 0


La comprobación confirma que las 398 filas del dataset de entrenamiento están completas: cero NaN en las 29 columnas de Camino A, y por extensión en las 7 de Camino B. El dataset queda validado y listo para el entrenamiento de modelos.

# 2. Entrenamiento de modelos

Se define el diccionario de modelos a comparar: los cinco clasificadores del benchmark original de Schmidt et al. (2018) con sus hiperparámetros fijos (Decision Tree, Random Forest, AdaBoost, LDA, kNN), una extensión propia con XGBoost, y dos baselines de referencia (predicción aleatoria y clase mayoritaria). Se define también `evaluar_loso()`, que aplica validación cruzada Leave-One-Subject-Out: en cada fold se entrena con 10 sujetos y se evalúa sobre el sujeto restante, reportando Accuracy y F1 macro promediados sobre los 11 folds.

In [7]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone
import xgboost as xgb
import numpy as np
import pandas as pd

SEED = 42

In [8]:

# Hiperparámetros fijos del paper (Schmidt et al., 2018) donde aplica; XGBoost es extensión propia, no réplica
modelos = {
    'DecisionTree':                  DecisionTreeClassifier(min_samples_split=20, random_state=SEED),
    'RandomForest':                  RandomForestClassifier(n_estimators=100, min_samples_split=20, random_state=SEED),
    'AdaBoost':                      AdaBoostClassifier(estimator=DecisionTreeClassifier(min_samples_split=20), n_estimators=100, random_state=SEED),
    'LDA':                           LinearDiscriminantAnalysis(),
    'kNN (k=9)':                     KNeighborsClassifier(n_neighbors=9),
    'XGBoost (extensión, no paper)': xgb.XGBClassifier(n_estimators=100, random_state=SEED, eval_metric='logloss'),
    'Aleatorio (baseline)':          DummyClassifier(strategy='uniform', random_state=SEED),
    'Clase mayoritaria (baseline)':  DummyClassifier(strategy='most_frequent'),
}

def evaluar_loso(X, y, groups, modelos):
    logo = LeaveOneGroupOut()
    resultados = []
    for nombre, modelo in modelos.items():
        acc_folds, f1_folds = [], []
        for train_idx, test_idx in logo.split(X, y, groups):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            m = clone(modelo)
            m.fit(X_train, y_train)
            y_pred = m.predict(X_test)
            acc_folds.append(accuracy_score(y_test, y_pred))
            f1_folds.append(f1_score(y_test, y_pred, average='macro'))
        resultados.append({
            'Modelo': nombre,
            'Accuracy_media': np.mean(acc_folds), 'Accuracy_std': np.std(acc_folds),
            'F1_macro_media': np.mean(f1_folds), 'F1_macro_std': np.std(f1_folds),
        })
    return pd.DataFrame(resultados).sort_values('F1_macro_media', ascending=False)


def evaluar_loso_detalle(X, y, groups, modelos_subset):
    logo = LeaveOneGroupOut()
    filas = []
    for nombre, modelo in modelos_subset.items():
        for train_idx, test_idx in logo.split(X, y, groups):
            sujeto_test = groups.iloc[test_idx].iloc[0]
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            m = clone(modelo)
            m.fit(X_train, y_train)
            y_pred = m.predict(X_test)
            filas.append({
                'Modelo': nombre,
                'Sujeto_test': sujeto_test,
                'N_minutos_test': len(y_test),
                'N_Estresado_test': int(y_test.sum()),
                'Accuracy': accuracy_score(y_test, y_pred),
                'F1_macro': f1_score(y_test, y_pred, average='macro'),
            })
    return pd.DataFrame(filas)

## 2.1. Camino A- Usando todas las features disponibles

Se evalúan los ocho modelos definidos en la Sección 2 sobre Camino A mediante LOSO, usando hiperparámetros fijos (sin optimización todavía, que se aborda en la Sección 3).

In [9]:
X_A = df_loso[columnas_camino_A]
y = df_loso['target']
groups = df_loso['ID_sujeto']

resultados_camino_A = evaluar_loso(X_A, y, groups, modelos)
print("Resultados Camino A (features completas):")
print(resultados_camino_A.to_string(index=False))

Resultados Camino A (features completas):
                       Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
                 RandomForest        0.867367      0.113315        0.827209      0.167115
                 DecisionTree        0.870161      0.113033        0.807964      0.198616
XGBoost (extensión, no paper)        0.871992      0.106292        0.798982      0.206343
                     AdaBoost        0.839786      0.112253        0.758806      0.207891
                    kNN (k=9)        0.779915      0.143607        0.664205      0.233130
                          LDA        0.787218      0.135497        0.634803      0.258880
         Aleatorio (baseline)        0.630666      0.022918        0.620499      0.023464
 Clase mayoritaria (baseline)        0.698598      0.010245        0.411258      0.003559


Random Forest obtiene el mejor F1 macro (0,8272), seguido de cerca por Decision Tree (0,8080) y XGBoost (0,7990). AdaBoost, kNN y LDA quedan claramente por debajo. Ambos baselines (aleatorio: 0,6205; clase mayoritaria: 0,4113) confirman que todos los modelos entrenados superan ampliamente el punto de referencia trivial, validando que Camino A contiene señal real para la tarea de clasificación de estrés.

### 2.1.1. Desglose de resultados por sujeto

Se desglosa el rendimiento por sujeto para los tres modelos más competitivos (RandomForest, XGBoost, AdaBoost), con el fin de identificar heterogeneidad oculta tras el promedio agregado de la Sección 2.1.

In [10]:
modelos_a_revisar = {k: modelos[k] for k in ['AdaBoost', 'XGBoost (extensión, no paper)', 'RandomForest']}
detalle_camino_A = evaluar_loso_detalle(X_A, y, groups, modelos_a_revisar)

for nombre in modelos_a_revisar:
    print(f"\n{nombre}:")
    print(detalle_camino_A[detalle_camino_A['Modelo'] == nombre]
          .drop(columns='Modelo')
          .sort_values('F1_macro')
          .to_string(index=False))


AdaBoost:
Sujeto_test  N_minutos_test  N_Estresado_test  Accuracy  F1_macro
         S9              37                11  0.621622  0.383333
         S3              37                11  0.702703  0.412698
         S8              36                11  0.722222  0.556650
         S2              35                10  0.800000  0.669366
        S16              36                11  0.833333  0.824675
        S15              36                11  0.833333  0.824675
         S4              36                11  0.916667  0.908085
        S14              36                11  0.944444  0.930769
        S17              37                12  0.945946  0.935315
        S13              37                11  0.945946  0.935315
         S7              35                10  0.971429  0.965986

XGBoost (extensión, no paper):
Sujeto_test  N_minutos_test  N_Estresado_test  Accuracy  F1_macro
         S3              37                11  0.702703  0.412698
         S2              35      

El desglose revela un patrón consistente en los tres modelos: **S3 es sistemáticamente el sujeto más difícil**, con el F1 macro más bajo en los tres casos (AdaBoost: 0,413; XGBoost: 0,413; RandomForest: 0,362). **S9 y S2** también aparecen recurrentemente entre los sujetos peor clasificados. En el extremo opuesto, S7, S4 y S13 se clasifican casi perfectamente en los tres modelos (F1 > 0,93). Esta heterogeneidad —oculta tras el promedio agregado de 2.1— sugiere que el rendimiento del modelo depende de características idiosincráticas de cada sujeto (posiblemente relacionadas con la magnitud de su respuesta fisiológica al estrés), más que de una limitación uniforme del clasificador.

## 2.2. Camino B Modelo ML sólo con variables relacionadas con HR

Se repite la misma evaluación LOSO de la Sección 2.1, restringiendo esta vez las variables de entrada a Camino B (7 columnas derivadas únicamente de HR), la configuración transferible al escenario real de wearables de consumo.

In [11]:
X_B = df_loso[columnas_camino_B]

resultados_camino_B = evaluar_loso(X_B, y, groups, modelos)
print("Resultados Camino B (solo HR):")
print(resultados_camino_B.to_string(index=False))

Resultados Camino B (solo HR):
                       Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
                 RandomForest        0.793955      0.107507        0.709955      0.202890
XGBoost (extensión, no paper)        0.788908      0.065393        0.705927      0.142507
                     AdaBoost        0.773529      0.099185        0.689889      0.174649
                    kNN (k=9)        0.779373      0.086557        0.668736      0.182579
                 DecisionTree        0.756919      0.145370        0.652784      0.224740
         Aleatorio (baseline)        0.630666      0.022918        0.620499      0.023464
                          LDA        0.756846      0.074738        0.576278      0.184866
 Clase mayoritaria (baseline)        0.698598      0.010245        0.411258      0.003559


Random Forest vuelve a ser el mejor modelo (F1 macro 0,7100), seguido de cerca por XGBoost (0,7059). El rendimiento de todos los modelos cae de forma notable respecto a Camino A (Random Forest: 0,8272 → 0,7100), confirmando el coste esperado de restringir la entrada a una única señal fisiológica. Aun así, todos los modelos superan claramente ambos baselines, indicando que HR por sí solo retiene señal útil para la tarea.

### 2.2.1. Desglose de resultados por sujeto

Se desglosa el rendimiento por sujeto para los cuatro modelos más relevantes de Camino B.

In [12]:
modelos_a_revisar_B = {k: modelos[k] for k in ['RandomForest', 'XGBoost (extensión, no paper)', 'LDA', 'DecisionTree']}
detalle_camino_B = evaluar_loso_detalle(X_B, y, groups, modelos_a_revisar_B)

for nombre in modelos_a_revisar_B:
    print(f"\n{nombre}:")
    print(detalle_camino_B[detalle_camino_B['Modelo'] == nombre]
          .drop(columns='Modelo')
          .sort_values('F1_macro')
          .to_string(index=False))


RandomForest:
Sujeto_test  N_minutos_test  N_Estresado_test  Accuracy  F1_macro
         S9              37                11  0.567568  0.362069
        S15              36                11  0.694444  0.409836
         S2              35                10  0.714286  0.416667
         S3              37                11  0.756757  0.750188
         S8              36                11  0.833333  0.758929
         S7              35                10  0.800000  0.782222
        S16              36                11  0.805556  0.797915
        S17              37                12  0.810811  0.805701
        S14              36                11  0.833333  0.824675
         S4              36                11  0.944444  0.934545
        S13              37                11  0.972973  0.966757

XGBoost (extensión, no paper):
Sujeto_test  N_minutos_test  N_Estresado_test  Accuracy  F1_macro
         S2              35                10  0.714286  0.416667
        S15              36  

S9, S15 y S2 emergen como los sujetos más difíciles en Random Forest (F1 macro entre 0,36 y 0,42), consistente con el patrón ya observado en Camino A para S9 y S2.

### 2.2.2. Análisis de matrices de confusión en sujetos difíciles

In [13]:
from sklearn.metrics import confusion_matrix

modelos_a_revisar_B_v2 = {k: modelos[k] for k in ['RandomForest', 'XGBoost (extensión, no paper)']}
sujetos_dificiles_B = ['S2', 'S9', 'S15']

for nombre_modelo, modelo in modelos_a_revisar_B_v2.items():
    print(f"\n=== {nombre_modelo} — Camino B v2 ===")
    for sujeto in sujetos_dificiles_B:
        train_idx = groups[groups != sujeto].index
        test_idx = groups[groups == sujeto].index
        m = clone(modelo)
        m.fit(X_B.loc[train_idx], y.loc[train_idx])
        y_pred = m.predict(X_B.loc[test_idx])
        cm = confusion_matrix(y.loc[test_idx], y_pred, labels=[0, 1])
        print(f"{sujeto} — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):")
        print(cm)


=== RandomForest — Camino B v2 ===
S2 — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):
[[25  0]
 [10  0]]
S9 — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):
[[21  5]
 [11  0]]
S15 — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):
[[25  0]
 [11  0]]

=== XGBoost (extensión, no paper) — Camino B v2 ===
S2 — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):
[[25  0]
 [10  0]]
S9 — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):
[[21  5]
 [ 8  3]]
S15 — matriz de confusión (filas=real, columnas=predicho, [No-estrés, Estresado]):
[[25  0]
 [10  1]]


El análisis de matrices de confusión sobre estos tres sujetos revela un hallazgo más grave que un simple F1 bajo: en S2 y S15, Random Forest **no detecta ni un solo minuto de Estrés** (0 verdaderos positivos), clasificando sistemáticamente al sujeto como no-estresado incluso en sus 10-11 minutos de TSST real. XGBoost mejora ligeramente este comportamiento en S9 (3 de 11 detectados) y S15 (1 de 10), sin introducir falsos positivos adicionales, pero mantiene el fallo total en S2. Esto indica que, para al menos S2, la señal de HR-only no contiene información discriminativa suficiente para distinguir su respuesta de estrés de su línea base — una limitación de fondo de Camino B, no solo un problema de calibración del modelo.

# 3. Optimización de modelos

Se define `cv_anidada()`, que aplica validación cruzada anidada: en el bucle externo (LOSO, un sujeto por fold) se realiza una búsqueda aleatoria de hiperparámetros (`RandomizedSearchCV`, 30 iteraciones) usando a su vez LOSO como esquema de validación interno sobre los 10 sujetos restantes de entrenamiento, optimizando F1 macro. Esto evita el sesgo optimista de ajustar hiperparámetros sobre el mismo conjunto que luego se evalúa. Se centra la optimización en Random Forest, el modelo más competitivo en la Sección 2, definiendo su espacio de búsqueda de hiperparámetros.

In [14]:
from sklearn.model_selection import RandomizedSearchCV, LeaveOneGroupOut
from scipy.stats import randint, uniform

def cv_anidada(X, y, groups, modelo_base, param_distributions, n_iter=30, seed=42):
    logo_externo = LeaveOneGroupOut()
    resultados = []
    for train_idx, test_idx in logo_externo.split(X, y, groups):
        sujeto_test = groups.iloc[test_idx].iloc[0]
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]

        logo_interno = list(LeaveOneGroupOut().split(X_train, y_train, groups_train))
        busqueda = RandomizedSearchCV(
            estimator=clone(modelo_base),
            param_distributions=param_distributions,
            n_iter=n_iter,
            cv=logo_interno,
            scoring='f1_macro',
            random_state=seed,
            n_jobs=-1,
        )
        busqueda.fit(X_train, y_train)
        y_pred = busqueda.best_estimator_.predict(X_test)

        resultados.append({
            'Sujeto_test': sujeto_test,
            'Accuracy': accuracy_score(y_test, y_pred),
            'F1_macro': f1_score(y_test, y_pred, average='macro'),
            'Mejores_hiperparametros': busqueda.best_params_,
        })
    return pd.DataFrame(resultados)

In [15]:
param_dist_rf = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 15),
    'min_samples_split': randint(2, 30),
    'min_samples_leaf': randint(1, 15),
    'max_features': ['sqrt', 'log2', None],
}
rf_base = RandomForestClassifier(random_state=SEED)

## 3.1. Camino A

Se aplica la optimización de hiperparámetros mediante CV anidada sobre Camino A.

In [16]:
resultados_nested_A = cv_anidada(X_A, y, groups, rf_base, param_dist_rf, n_iter=30, seed=SEED)
print("=== CV anidada — Camino A (RandomForest) ===")
print(resultados_nested_A[['Sujeto_test', 'Accuracy', 'F1_macro']].sort_values('F1_macro').to_string(index=False))
print(f"\nF1 macro medio: {resultados_nested_A['F1_macro'].mean():.4f} (± {resultados_nested_A['F1_macro'].std():.4f})")

=== CV anidada — Camino A (RandomForest) ===
Sujeto_test  Accuracy  F1_macro
         S3  0.702703  0.412698
         S2  0.828571  0.732143
        S14  0.833333  0.824675
        S15  0.833333  0.824675
        S16  0.833333  0.824675
         S8  0.888889  0.851852
         S9  0.891892  0.853175
        S17  0.891892  0.862963
        S13  0.972973  0.966757
         S4  0.972222  0.968057
         S7  1.000000  1.000000

F1 macro medio: 0.8292 (± 0.1589)


El F1 macro medio tras optimización (0,8292 ± 0,1589) es prácticamente idéntico al obtenido con hiperparámetros fijos en la Sección 2.1 (0,8272), sin mejora sustancial. S3 sigue siendo, con diferencia, el sujeto más difícil (F1 = 0,413), confirmando que su dificultad no se debe a una mala elección de hiperparámetros sino a una característica propia de ese sujeto — la optimización no la resuelve.

## 3.2. Camino B

Se aplica la misma optimización de hiperparámetros mediante CV anidada sobre Camino B.

In [17]:
resultados_nested_B = cv_anidada(X_B, y, groups, rf_base, param_dist_rf, n_iter=30, seed=SEED)
print("=== CV anidada — Camino B (RandomForest) ===")
print(resultados_nested_B[['Sujeto_test', 'Accuracy', 'F1_macro']].sort_values('F1_macro').to_string(index=False))
print(f"\nF1 macro medio: {resultados_nested_B['F1_macro'].mean():.4f} (± {resultados_nested_B['F1_macro'].std():.4f})")

=== CV anidada — Camino B (RandomForest) ===
Sujeto_test  Accuracy  F1_macro
         S9  0.567568  0.362069
         S2  0.714286  0.416667
        S15  0.722222  0.500000
         S8  0.777778  0.645320
         S3  0.783784  0.775758
        S16  0.805556  0.797915
        S17  0.810811  0.805701
        S14  0.833333  0.824675
         S4  0.944444  0.937500
        S13  0.972973  0.966757
         S7  1.000000  1.000000

F1 macro medio: 0.7302 (± 0.2207)


El F1 macro medio (0,7302 ± 0,2207) apenas mejora respecto a los hiperparámetros fijos de la Sección 2.2 (0,7100), y la dispersión entre sujetos aumenta respecto a Camino A. El patrón de sujetos difíciles cambia parcialmente: S9 pasa a ser el peor (F1 = 0,362), seguido de S2 y S15 — los mismos tres sujetos ya identificados en el desglose de 2.2.1, confirmando que la dificultad es específica de esos sujetos y no un artefacto de la elección de hiperparámetros o del modelo.

## 3.3. Efecto del balanceo de clases sobre el rendimiento

Se evalúa si forzar `class_weight='balanced'` en Random Forest (con los hiperparámetros fijos de la Sección 2) mejora el rendimiento respecto al modelo sin balanceo, dado el desbalance moderado de clases (2,3:1) identificado en 1.2.

In [18]:
rf_balanceado = RandomForestClassifier(n_estimators=100, min_samples_split=20, class_weight='balanced', random_state=SEED)

resultados_balanceo_A = evaluar_loso(X_A, y, groups, {'RandomForest (balanceado)': rf_balanceado})
print("Camino A — con balanceo de clases:")
print(resultados_balanceo_A.to_string(index=False))

resultados_balanceo_B = evaluar_loso(X_B, y, groups, {'RandomForest (balanceado)': rf_balanceado})
print("\nCamino B — con balanceo de clases:")
print(resultados_balanceo_B.to_string(index=False))

Camino A — con balanceo de clases:
                   Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
RandomForest (balanceado)        0.854797      0.128945        0.807427      0.195233

Camino B — con balanceo de clases:
                   Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
RandomForest (balanceado)        0.803984      0.118555        0.738177      0.197665


El efecto del balanceo de clases no es uniforme entre caminos. En Camino B mejora el F1 macro (0,7100 → 0,7382), coherente con que una señal más débil (solo HR) se beneficia de una corrección explícita del desbalance. En Camino A, sin embargo, el balanceo **empeora** ligeramente el resultado (0,8272 → 0,8074) — con una señal más rica, forzar `class_weight='balanced'` introduce una penalización que no compensa, posiblemente porque el modelo sin balancear ya discrimina bien ambas clases y el ajuste añade ruido en la frontera de decisión.

## 3.4. Camino B con CV anidada + balanceo

Se combina la optimización de hiperparámetros mediante CV anidada con la posibilidad de balanceo, incluyendo `class_weight` (`'balanced'` o `None`) como un hiperparámetro más dentro del espacio de búsqueda, en lugar de forzarlo de antemano.

In [19]:
param_dist_rf_balanceado = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 15),
    'min_samples_split': randint(2, 30),
    'min_samples_leaf': randint(1, 15),
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced', None],
}
rf_base_balanceable = RandomForestClassifier(random_state=SEED)

resultados_nested_B_balanceado = cv_anidada(X_B, y, groups, rf_base_balanceable, param_dist_rf_balanceado, n_iter=30, seed=SEED)
print("=== CV anidada + balanceo — Camino B (RandomForest) ===")
print(resultados_nested_B_balanceado[['Sujeto_test', 'Accuracy', 'F1_macro']].sort_values('F1_macro').to_string(index=False))
print(f"\nF1 macro medio: {resultados_nested_B_balanceado['F1_macro'].mean():.4f} (± {resultados_nested_B_balanceado['F1_macro'].std():.4f})")

=== CV anidada + balanceo — Camino B (RandomForest) ===
Sujeto_test  Accuracy  F1_macro
         S9  0.540541  0.398086
        S15  0.694444  0.409836
         S2  0.714286  0.416667
         S8  0.722222  0.500000
         S3  0.756757  0.750188
         S7  0.771429  0.761905
        S17  0.783784  0.779762
        S16  0.805556  0.797915
        S14  0.833333  0.824675
         S4  0.888889  0.875000
        S13  1.000000  1.000000

F1 macro medio: 0.6831 (± 0.2124)


El resultado combinado (F1 macro = 0,6831 ± 0,2124) es **peor** que la CV anidada sin balanceo forzado de 3.2 (0,7302) y que el balanceo forzado con hiperparámetros fijos de 3.3 (0,7382). Es decir, ni la optimización de hiperparámetros ni el balanceo de clases, por separado, perjudican el rendimiento — pero su combinación en este espacio de búsqueda sí lo hace. Una explicación plausible es que permitir que `RandomizedSearchCV` elija libremente entre balanceado y no balanceado, junto con el resto de hiperparámetros, diluye la señal de optimización en un espacio de búsqueda más amplio con el mismo número de iteraciones (30) y un conjunto de entrenamiento pequeño (10 sujetos por fold interno) — el balanceo forzado y fijo de 3.3 resulta, en la práctica, más efectivo que dejarlo como una opción más a optimizar.

## 3.5. Test de Wilcoxon pareado: Camino A vs. Camino B

In [20]:
# ======================================================================================================
# TEST DE WILCOXON PAREADO: Camino A vs. Camino B (CV anidada, RandomForest)
# ======================================================================================================
comparacion_A_vs_B = resultados_nested_A[['Sujeto_test', 'F1_macro']].merge(
    resultados_nested_B[['Sujeto_test', 'F1_macro']],
    on='Sujeto_test', suffixes=('_A', '_B')
)
comparacion_A_vs_B['Delta_F1_macro'] = (
    comparacion_A_vs_B['F1_macro_A'] - comparacion_A_vs_B['F1_macro_B']
)
print(comparacion_A_vs_B.sort_values('Delta_F1_macro').to_string(index=False))
print(f"\nSujetos donde Camino A supera a Camino B: {(comparacion_A_vs_B['Delta_F1_macro'] > 0).sum()} de {len(comparacion_A_vs_B)}")
print(f"Delta medio: {comparacion_A_vs_B['Delta_F1_macro'].mean():.4f}")
print(f"Delta mediano: {comparacion_A_vs_B['Delta_F1_macro'].median():.4f}")

from scipy.stats import wilcoxon
stat, p = wilcoxon(comparacion_A_vs_B['F1_macro_A'], comparacion_A_vs_B['F1_macro_B'])
print(f"\nWilcoxon (pareado, Camino A vs. Camino B): estadístico={stat:.3f}, p={p:.4f}")

Sujeto_test  F1_macro_A  F1_macro_B  Delta_F1_macro
         S3    0.412698    0.775758       -0.363059
        S13    0.966757    0.966757        0.000000
        S14    0.824675    0.824675        0.000000
         S7    1.000000    1.000000        0.000000
        S16    0.824675    0.797915        0.026760
         S4    0.968057    0.937500        0.030557
        S17    0.862963    0.805701        0.057262
         S8    0.851852    0.645320        0.206532
         S2    0.732143    0.416667        0.315476
        S15    0.824675    0.500000        0.324675
         S9    0.853175    0.362069        0.491106

Sujetos donde Camino A supera a Camino B: 7 de 11
Delta medio: 0.0990
Delta mediano: 0.0306

Wilcoxon (pareado, Camino A vs. Camino B): estadístico=7.000, p=0.1484


El test de Wilcoxon pareado (n=11, estadístico=7,0, p=0,1484) **no permite afirmar con significación estadística** que Camino A supere a Camino B, pese a que la comparación descriptiva es favorable a Camino A en 7 de los 11 sujetos (delta medio=0,099, delta mediano=0,031). El caso más discordante es S3: es el sujeto más difícil en Camino A (F1=0,413) pero mejora sustancialmente en Camino B (F1=0,776) — el único sujeto donde restringir a HR-only ayuda en vez de perjudicar. Como en la ablación de normalización (Sección 5), un p-valor no significativo con esta no equivale a "no hay diferencia real"; solo indica que la muestra es insuficiente para demostrarla con confianza.

## 3.6. Matriz de confusión de la configuración final

In [21]:
from sklearn.metrics import confusion_matrix

def confusion_matrix_por_sujeto(X, y, groups, modelo):
    logo = LeaveOneGroupOut()
    resultados = {}
    for train_idx, test_idx in logo.split(X, y, groups):
        sujeto_test = groups.iloc[test_idx].iloc[0]
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        m = clone(modelo)
        m.fit(X_train, y_train)
        y_pred = m.predict(X_test)
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        resultados[sujeto_test] = cm
    return resultados

# Camino B, configuración final balanceada
X_B = df_loso[columnas_camino_B]
cm_balanceado = confusion_matrix_por_sujeto(X_B, y, groups, rf_balanceado)

for sujeto in cm_balanceado:  # Ver resultados de todos los sujetos, no solo los difíciles
    cm = cm_balanceado[sujeto]
    print(f"{sujeto}: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

S13: TN=26, FP=0, FN=0, TP=11
S14: TN=19, FP=6, FN=0, TP=11
S15: TN=25, FP=0, FN=10, TP=1
S16: TN=18, FP=7, FN=0, TP=11
S17: TN=17, FP=8, FN=0, TP=12
S2: TN=25, FP=0, FN=10, TP=0
S3: TN=17, FP=9, FN=0, TP=11
S4: TN=22, FP=3, FN=0, TP=11
S7: TN=17, FP=8, FN=0, TP=10
S8: TN=24, FP=1, FN=0, TP=11
S9: TN=20, FP=6, FN=10, TP=1


El balanceo de clases no resuelve el problema de sensibilidad comprometida en los tres sujetos identificados: S2 mantiene recall nulo (0/10) de forma idéntica a la configuración sin balancear, confirmando que la limitación no es de umbral de decisión sino de ausencia de señal discriminativa en HR para ese sujeto. S9 y S15 pasan de recall prácticamente nulo a 1/11 (9,1%) cada uno, una mejora marginal en magnitud absoluta, con distinto coste: S9 introduce 6 falsos positivos adicionales, mientras que S15 mejora sin coste en falsos positivos

## 3.7. Conclusiones optimización de camino A y camino B

**Camino A supera consistentemente a Camino B, como cabía esperar.** En CV anidada con RandomForest, Camino A (todas las señales disponibles) alcanza un F1 macro medio de 0,8292 (± 0,1589) frente a 0,7302 (± 0,2207) de Camino B (solo HR). La misma brecha se mantiene con hiperparámetros fijos y balanceo forzado (Camino A: 0,8074 vs. Camino B: 0,7382). Esta diferencia es el coste esperado y ya justificado de la restricción de dominio: Camino B no es una versión empobrecida sin motivo, es la única compatible con las señales que realmente entrega el wearable de consumo (Xiaomi S4) en la validación n-of-1. Camino A actúa como cota superior de referencia — el rendimiento alcanzable si se dispusiera de EDA/TEMP/HRV fiables —, no como candidato a modelo final.

**El efecto del balanceo de clases no es monótono, y el hallazgo más interesante va en contra de la intuición inicial.** Se probaron tres configuraciones sobre Camino B:

| Configuración | F1 macro medio |
|---|---|
| CV anidada, sin balanceo forzado (class_weight fuera del espacio de búsqueda) | 0,7302 (± 0,2207) |
| Hiperparámetros fijos + balanceo forzado (`class_weight='balanced'`) | 0,7382 (± 0,1977) |
| CV anidada + balanceo como hiperparámetro buscado | 0,6831 (± 0,2124) |

Incluir `class_weight` como una opción más dentro de la búsqueda aleatoria de hiperparámetros (`RandomizedSearchCV`, n_iter=30) da el peor resultado de las tres, por debajo incluso de no tocar el balanceo. La explicación más plausible no es que el balanceo perjudique al modelo — la configuración de hiperparámetros fijos con balanceo forzado es la que mejor resultado da de las tres —, sino que ampliar el espacio de búsqueda con una dimensión categórica adicional, sin aumentar el presupuesto de iteraciones (n_iter=30 fijo), diluye la capacidad de la búsqueda aleatoria para converger a una buena combinación del resto de hiperparámetros. Es decir: el problema no es el balanceo en sí, es la combinación de un espacio de búsqueda más grande con un presupuesto de exploración constante.

**Decisión adoptada:** se descarta el balanceo como hiperparámetro sujeto a búsqueda, y se retiene la configuración de hiperparámetros fijos con balanceo forzado (`n_estimators=100, min_samples_split=20, class_weight='balanced'`) para Camino B — que es, de hecho, la configuración con la que ya está entrenado el modelo final (sección 7). Esto añade una justificación explícita a una elección que hasta ahora estaba implícita en el código.

**Matriz de confusión de la configuración adoptada (3.6)**: el balanceo mejora la sensibilidad de forma marginal en S9 y S15 (recall 0%→9,1% en ambos, con distinto coste en falsos positivos) y no la mejora en absoluto en S2 (recall 0/10 en ambas configuraciones). La ganancia de F1 macro agregado (0,7100→0,7382) no debe leerse, por tanto, como una corrección generalizada del problema de sensibilidad comprometida en estos tres sujetos, sino como una mejora parcial y desigual entre ellos — la misma alta varianza entre sujetos que se cuantifica a continuación como limitación no es solo una cifra de dispersión estadística, es precisamente lo que hace que una misma decisión de diseño (balancear) beneficie de forma tan distinta a S2, S9 y S15

**Limitación a declarar:** con n=11 sujetos en LOSO, la desviación estándar del F1 macro entre configuraciones (0,16–0,22) es del mismo orden de magnitud que las propias diferencias entre configuraciones (0,05–0,10). La comparación Camino A vs. Camino B ya cuenta con test de Wilcoxon pareado (Sección 3.5: p=0,1484, no significativo). Las comparaciones entre las tres configuraciones de balanceo, sin embargo, no se han sometido a ningún test estadístico pareado — con esta n, es probable que las diferencias observadas (0,05–0,10 en F1 macro) tampoco alcanzaran significación, pero esto no se ha verificado. Estas comparaciones deben presentarse como "diferencia observada en la media", no como "diferencia demostrada".

# 4. Estudio de ablación: ¿aportan mejora las ventanas de rolling y lag de 3 minutos?

En la Sección 1.3 se excluyeron las columnas de rolling y lag de 3 minutos por argumento de diseño (solapamiento con el lag de 1-2 minutos y el rolling ya cerrado de 5 minutos, y coste de dimensionalidad sobre un dataset de 398 filas). Esta sección valida esa decisión empíricamente, comparando el rendimiento de Camino A y Camino B con y sin las ventanas de 3 minutos.

## 4.1. Camino A y Camino B con ventanas de 3 minutos incorporadas por defecto

In [22]:
# ======================================================================================================
# COMPROBACIÓN EMPÍRICA: ¿aportan mejora las ventanas de 3 minutos si se incluyen por defecto?
# ======================================================================================================

# Camino A y B CON ventanas de 3 minutos (sin excluir '_3m')
columnas_camino_A_con_3m = [
    c for c in df_loso.columns
    if c.startswith(prefijos_camino_A) and '_norm' in c
]
columnas_camino_B_con_3m = [
    c for c in df_loso.columns
    if c.startswith(prefijos_camino_B) and '_norm' in c
]

print(f"Camino A con 3m: {len(columnas_camino_A_con_3m)} columnas (vs. {len(columnas_camino_A)} sin 3m)")
print(f"Camino B con 3m: {len(columnas_camino_B_con_3m)} columnas (vs. {len(columnas_camino_B)} sin 3m)")

X_A_con_3m = df_loso[columnas_camino_A_con_3m]
X_B_con_3m = df_loso[columnas_camino_B_con_3m]

resultados_A_con_3m = evaluar_loso(X_A_con_3m, y, groups, {'RandomForest': modelos['RandomForest']})
resultados_B_con_3m = evaluar_loso(X_B_con_3m, y, groups, {'RandomForest': modelos['RandomForest']})

print("\nCamino A — con ventanas de 3m:")
print(resultados_A_con_3m.to_string(index=False))
print(f"(Referencia sin 3m: F1 macro = {resultados_camino_A[resultados_camino_A['Modelo']=='RandomForest']['F1_macro_media'].values[0]:.4f})")

print("\nCamino B — con ventanas de 3m:")
print(resultados_B_con_3m.to_string(index=False))
print(f"(Referencia sin 3m: F1 macro = {resultados_camino_B[resultados_camino_B['Modelo']=='RandomForest']['F1_macro_media'].values[0]:.4f})")

Camino A con 3m: 41 columnas (vs. 29 sin 3m)
Camino B con 3m: 10 columnas (vs. 7 sin 3m)

Camino A — con ventanas de 3m:
      Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
RandomForest        0.857334      0.119481        0.815888      0.170275
(Referencia sin 3m: F1 macro = 0.8272)

Camino B — con ventanas de 3m:
      Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
RandomForest        0.788908      0.104492        0.698788      0.199739
(Referencia sin 3m: F1 macro = 0.7100)


## 4.2. Camino B + rolling/lag de 3 minutos + velocidad de cambio

In [23]:
# ======================================================================================================
# COMPROBACIÓN EMPÍRICA: Camino B + ventana de 3m + feature de velocidad de cambio
# ======================================================================================================

# Se parte de Camino B (sin 3m) y se añaden las columnas de 3 minutos ya presentes en el dataset maestro
columnas_3m_hr = [c for c in df_loso.columns if c.startswith('HR_') and '_norm' in c and '_3m' in c]

# Feature de velocidad de cambio: diferencia de HR_mean_norm respecto al minuto anterior, por sujeto
df_loso['HR_mean_norm_velocidad'] = df_loso.groupby('ID_sujeto')['HR_mean_norm'].diff().fillna(0)

columnas_camino_B_extendido = columnas_camino_B + columnas_3m_hr + ['HR_mean_norm_velocidad']
print(f"Camino B extendido (3m + velocidad): {len(columnas_camino_B_extendido)} columnas")

X_B_extendido = df_loso[columnas_camino_B_extendido]

resultados_B_extendido = evaluar_loso(X_B_extendido, y, groups, {'RandomForest': modelos['RandomForest']})
print("\nCamino B extendido — resultado:")
print(resultados_B_extendido.to_string(index=False))
print(f"(Referencia Camino B base: F1 macro = {resultados_camino_B[resultados_camino_B['Modelo']=='RandomForest']['F1_macro_media'].values[0]:.4f})")

Camino B extendido (3m + velocidad): 11 columnas

Camino B extendido — resultado:
      Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
RandomForest        0.783858      0.104908        0.694865      0.198059
(Referencia Camino B base: F1 macro = 0.7100)


In [24]:
# Se elimina inmediatamente la columna experimental de velocidad de cambio, para que no contamine
# ninguna sección posterior que seleccione columnas por prefijo/patrón (p. ej. Camino A + ACC en la Sección 6).
df_loso = df_loso.drop(columns=['HR_mean_norm_velocidad'], errors='ignore')

## 4.3. Conclusión

| Configuración | F1 macro (sin 3m) | F1 macro (con 3m) |
|---|---|---|
| Camino A, incorporando ventanas de 3m por defecto | 0,8272 | 0,8159 |
| Camino B, incorporando ventanas de 3m por defecto | 0,7100 | 0,6988 |
| Camino B + rolling_3m + lag_3m + velocidad de cambio | 0,7100 | 0,6949 |

En las tres comprobaciones, añadir información de la ventana de 3 minutos produce una degradación leve y consistente del rendimiento (entre -0,01 y -0,02 en F1 macro), nunca una mejora. Esto respalda empíricamente la exclusión adoptada en 1.3, más allá del argumento de no-redundancia por solapamiento de escalas temporales.

**Decisión:** se mantiene la exclusión de las ventanas de 3 minutos en Camino A y Camino B.

# 5. Estudio de ablación: ¿aporta la normalización por sujeto (HR_mean_norm) mejora real de rendimiento?

Random Forest es invariante a la escala — no necesita normalización por motivos algorítmicos (los umbrales de división se ajustan proporcionalmente a cualquier reescalado monótono). Pero `HR_mean_norm` no es un simple reescalado: es un z-score contra el baseline personal de cada sujeto (condición Relajado, ver `WESAD_preprocesamiento.ipynb`), pensado para que el modelo generalice entre sujetos con líneas base de HR distintas — un problema de generalización entre individuos, no un requisito matemático del algoritmo.

Esta ablación lo comprueba empíricamente: misma partición LOSO, mismo target, misma arquitectura de Camino B (7 columnas, misma estructura temporal), sustituyendo únicamente las columnas normalizadas por sus equivalentes en HR bruto (bpm).

Se construyen las columnas equivalentes a Camino B pero sin normalizar, partiendo del HR en bruto (`HR_mean`, en bpm). Los rolling de 5 y 10 minutos ya existen en el dataset maestro en su versión bruta (se generaron en paralelo a los normalizados durante el preprocesamiento), pero los lags de 1 y 2 minutos no — el preprocesamiento solo los calculó sobre la serie normalizada — por lo que se recalculan aquí replicando la misma lógica temporal (agrupado por sujeto, ordenado por minuto).

In [25]:
# Columnas en bruto (HR_mean, sin normalizar). Los rolling ya existen en el dataset maestro
# (se generaron en paralelo a los normalizados en el preprocesamiento). Los lags en bruto NO
# existen — el preprocesamiento solo generó lags sobre la serie normalizada — así que se
# recalculan aquí replicando la misma lógica temporal (agrupado por sujeto, ordenado por
# minuto_id) que se usó para construir HR_mean_norm_lag_1m_norm / _2m_norm.

df_loso_raw = df_loso.sort_values(['ID_sujeto', 'minuto_id']).copy()
df_loso_raw['HR_mean_lag_1m'] = df_loso_raw.groupby('ID_sujeto')['HR_mean'].shift(1)
df_loso_raw['HR_mean_lag_2m'] = df_loso_raw.groupby('ID_sujeto')['HR_mean'].shift(2)

columnas_camino_B_bruto = [
    'HR_mean',
    'HR_mean_roll_mean_5m', 'HR_mean_roll_std_5m',
    'HR_mean_roll_mean_10m', 'HR_mean_roll_std_10m',
    'HR_mean_lag_1m', 'HR_mean_lag_2m',
]

# El shift() introduce NaN en los primeros 1-2 minutos de cada sujeto (sin lag disponible)
# — mismo criterio de limpieza que el ya aplicado al dataset normalizado original (ver 1.4).
filas_antes = len(df_loso_raw)
df_loso_raw = df_loso_raw.dropna(subset=columnas_camino_B_bruto)
print(f"Filas antes de eliminar NaN por lag: {filas_antes} -> después: {len(df_loso_raw)}")

X_B_bruto = df_loso_raw[columnas_camino_B_bruto]
y_bruto = df_loso_raw['target']
groups_bruto = df_loso_raw['ID_sujeto']

assert not X_B_bruto.isna().any().any(), "No debería haber NaN en X_B_bruto"
assert groups_bruto.nunique() == 11, "Deberían seguir siendo 11 sujetos tras el dropna"
print(f"\nColumnas Camino B (bruto, sin normalizar): {list(columnas_camino_B_bruto)}")
print(f"Filas finales: {len(X_B_bruto)}, sujetos: {groups_bruto.nunique()}")

Filas antes de eliminar NaN por lag: 398 -> después: 376

Columnas Camino B (bruto, sin normalizar): ['HR_mean', 'HR_mean_roll_mean_5m', 'HR_mean_roll_std_5m', 'HR_mean_roll_mean_10m', 'HR_mean_roll_std_10m', 'HR_mean_lag_1m', 'HR_mean_lag_2m']
Filas finales: 376, sujetos: 11


## 5.1. LOSO simple (mismos modelos que 2.2, para comparación directa)

Se evalúan los mismos ocho modelos de la Sección 2.2 sobre las columnas de HR en bruto construidas justo antes de esta subsección, replicando por bpm sin normalizar las columnas equivalentes de Camino B (incluyendo los lags, recalculados aquí porque el preprocesamiento solo los generó sobre la serie normalizada).

In [26]:
resultados_camino_B_bruto = evaluar_loso(X_B_bruto, y_bruto, groups_bruto, modelos)
print("Resultados Camino B (HR en bruto, sin normalizar):")
print(resultados_camino_B_bruto.to_string(index=False))

Resultados Camino B (HR en bruto, sin normalizar):
                       Modelo  Accuracy_media  Accuracy_std  F1_macro_media  F1_macro_std
                     AdaBoost        0.763324      0.169040        0.720842      0.195329
                 RandomForest        0.766150      0.205355        0.715859      0.250478
                          LDA        0.816126      0.160135        0.715274      0.261610
XGBoost (extensión, no paper)        0.744665      0.174555        0.701733      0.200460
                 DecisionTree        0.726497      0.225569        0.688035      0.246954
         Aleatorio (baseline)        0.670257      0.009408        0.662608      0.010308
                    kNN (k=9)        0.749564      0.154044        0.656912      0.222907
 Clase mayoritaria (baseline)        0.680962      0.010653        0.405078      0.003781


## 5.2. Comparación directa: normalizado vs. bruto (LOSO simple)

Se cruzan los resultados de Camino B normalizado (Sección 2.2) y Camino B en bruto (5.1) por modelo, calculando la diferencia de F1 macro para cuantificar el efecto de la normalización antes de pasar a la comparación más rigurosa con CV anidada.

In [27]:
comparacion_bruto_norm = resultados_camino_B[['Modelo', 'Accuracy_media', 'F1_macro_media']].merge(
    resultados_camino_B_bruto[['Modelo', 'Accuracy_media', 'F1_macro_media']],
    on='Modelo', suffixes=('_normalizado', '_bruto')
)
comparacion_bruto_norm['Delta_F1_macro'] = (
    comparacion_bruto_norm['F1_macro_media_bruto'] - comparacion_bruto_norm['F1_macro_media_normalizado']
)
print(comparacion_bruto_norm.sort_values('Delta_F1_macro').to_string(index=False))

                       Modelo  Accuracy_media_normalizado  F1_macro_media_normalizado  Accuracy_media_bruto  F1_macro_media_bruto  Delta_F1_macro
                    kNN (k=9)                    0.779373                    0.668736              0.749564              0.656912       -0.011825
 Clase mayoritaria (baseline)                    0.698598                    0.411258              0.680962              0.405078       -0.006179
XGBoost (extensión, no paper)                    0.788908                    0.705927              0.744665              0.701733       -0.004194
                 RandomForest                    0.793955                    0.709955              0.766150              0.715859        0.005904
                     AdaBoost                    0.773529                    0.689889              0.763324              0.720842        0.030953
                 DecisionTree                    0.756919                    0.652784              0.726497              0.6

## 5.3. CV anidada — RandomForest, Camino B bruto (comparable directamente a 3.2)

Misma búsqueda de hiperparámetros (`param_dist_rf`) y misma función `cv_anidada` ya usada en 3.1/3.2 — única variable que cambia es el conjunto de features (bruto vs. normalizado).

In [28]:
resultados_nested_B_bruto = cv_anidada(X_B_bruto, y_bruto, groups_bruto, rf_base, param_dist_rf, n_iter=30, seed=SEED)
print("=== CV anidada — Camino B bruto (RandomForest) ===")
print(resultados_nested_B_bruto[['Sujeto_test', 'Accuracy', 'F1_macro']].sort_values('F1_macro').to_string(index=False))
print(f"\nF1 macro medio (bruto): {resultados_nested_B_bruto['F1_macro'].mean():.4f} (± {resultados_nested_B_bruto['F1_macro'].std():.4f})")
print(f"F1 macro medio (normalizado, referencia 3.2): {resultados_nested_B['F1_macro'].mean():.4f} (± {resultados_nested_B['F1_macro'].std():.4f})")

=== CV anidada — Camino B bruto (RandomForest) ===
Sujeto_test  Accuracy  F1_macro
         S9  0.457143  0.313725
         S2  0.484848  0.326531
        S15  0.558824  0.507246
         S8  0.676471  0.568627
         S4  0.764706  0.640212
        S13  0.742857  0.739454
         S3  0.771429  0.761905
         S7  0.939394  0.923611
        S17  0.942857  0.933712
        S16  1.000000  1.000000
        S14  1.000000  1.000000

F1 macro medio (bruto): 0.7014 (± 0.2523)
F1 macro medio (normalizado, referencia 3.2): 0.7302 (± 0.2207)


### 5.3.1. Test de Wilcoxon pareado

In [29]:
comparacion_pareada = resultados_nested_B[['Sujeto_test', 'F1_macro']].merge(
    resultados_nested_B_bruto[['Sujeto_test', 'F1_macro']],
    on='Sujeto_test', suffixes=('_normalizado', '_bruto')
)
comparacion_pareada['Delta_F1_macro'] = (
    comparacion_pareada['F1_macro_bruto'] - comparacion_pareada['F1_macro_normalizado']
)
print(comparacion_pareada.sort_values('Delta_F1_macro').to_string(index=False))
print(f"\nSujetos donde bruto empeora: {(comparacion_pareada['Delta_F1_macro'] < 0).sum()} de {len(comparacion_pareada)}")
print(f"Delta medio: {comparacion_pareada['Delta_F1_macro'].mean():.4f}")
print(f"Delta mediano: {comparacion_pareada['Delta_F1_macro'].median():.4f}")

from scipy.stats import wilcoxon
stat, p = wilcoxon(comparacion_pareada['F1_macro_normalizado'], comparacion_pareada['F1_macro_bruto'])
print(f"\nWilcoxon (pareado, normalizado vs. bruto): estadístico={stat:.3f}, p={p:.4f}")

Sujeto_test  F1_macro_normalizado  F1_macro_bruto  Delta_F1_macro
         S4              0.937500        0.640212       -0.297288
        S13              0.966757        0.739454       -0.227302
         S2              0.416667        0.326531       -0.090136
         S8              0.645320        0.568627       -0.076693
         S7              1.000000        0.923611       -0.076389
         S9              0.362069        0.313725       -0.048343
         S3              0.775758        0.761905       -0.013853
        S15              0.500000        0.507246        0.007246
        S17              0.805701        0.933712        0.128011
        S14              0.824675        1.000000        0.175325
        S16              0.797915        1.000000        0.202085

Sujetos donde bruto empeora: 7 de 11
Delta medio: -0.0288
Delta mediano: -0.0483

Wilcoxon (pareado, normalizado vs. bruto): estadístico=25.000, p=0.5195


### 5.3.2. Exploración de causa fisiológica: distancia al HR basal poblacional

In [30]:
baseline_por_sujeto = (
    df_loso[df_loso['Estado_Emocional'] == 'Relajado']
    .groupby('ID_sujeto')['HR_mean']
    .mean()
    .rename('HR_basal_relajado')
)

comparacion_con_baseline = comparacion_pareada.merge(
    baseline_por_sujeto, left_on='Sujeto_test', right_index=True
)
media_poblacional = baseline_por_sujeto.mean()
comparacion_con_baseline['Distancia_a_media_poblacional'] = (
    comparacion_con_baseline['HR_basal_relajado'] - media_poblacional
).abs()

print(comparacion_con_baseline[
    ['Sujeto_test', 'HR_basal_relajado', 'Distancia_a_media_poblacional', 'Delta_F1_macro']
].sort_values('Distancia_a_media_poblacional').to_string(index=False))

print(f"\nCorrelación (distancia a la media poblacional vs. Delta_F1_macro): "
      f"{comparacion_con_baseline['Distancia_a_media_poblacional'].corr(comparacion_con_baseline['Delta_F1_macro']):.3f}")

Sujeto_test  HR_basal_relajado  Distancia_a_media_poblacional  Delta_F1_macro
        S14          72.538211                       1.497448        0.175325
         S8          69.541456                       1.499306       -0.076693
         S7          69.389965                       1.650797       -0.076389
        S16          67.849158                       3.191604        0.202085
         S9          74.821767                       3.781005       -0.048343
        S17          66.749316                       4.291446        0.128011
         S2          76.528219                       5.487457       -0.090136
        S15          79.865254                       8.824492        0.007246
         S4          61.181886                       9.858876       -0.297288
         S3          58.452193                      12.588569       -0.013853
        S13          84.530958                      13.490196       -0.227302

Correlación (distancia a la media poblacional vs. Delta_F1_macr

## 5.4. Conclusión

**Decisión: se mantiene la normalización por sujeto (HR_mean_norm) en el modelo final de Camino B.**

**Resultados de la ablación (CV anidada, RandomForest, n=11 sujetos LOSO):**
- F1 macro medio normalizado: 0,7302 (± 0,2207)
- F1 macro medio bruto (sin normalizar): 0,7014 (± 0,2523)
- 7 de 11 sujetos empeoran al quitar la normalización; delta mediano por sujeto: -0,0483
- Test de Wilcoxon pareado: estadístico=25,0, p=0,52 — no significativo

**El test de Wilcoxon no permite afirmar que la normalización mejore el rendimiento de forma estadísticamente significativa** — con n=11 sujetos, el test tiene potencia insuficiente para detectar un efecto de esta magnitud. Un p-valor no significativo aquí no equivale a "no hay diferencia"; solo indica que no puede demostrarse con esta muestra.

**Explicación fisiológica explorada (parcial, no concluyente):** se comprobó si la caída de rendimiento sin normalizar se concentra en los sujetos cuyo HR basal (condición Relajado) está más alejado de la media poblacional del grupo LOSO — hipótesis consistente con que la normalización corrige diferencias interindividuales de línea base. Los dos sujetos con mayor caída (S4, Δ=-0,297; S13, Δ=-0,227) tienen, en efecto, de las mayores distancias a la media poblacional (9,86 y 13,49 bpm respectivamente). Sin embargo, la correlación entre distancia al HR basal medio y Δ F1_macro es moderada y no significativa (r=-0,53, p=0,09, n=11), y dos sujetos (S3, S15) contradicen el patrón de forma clara: S3 tiene la segunda mayor distancia (12,6 bpm) pero un delta casi nulo (-0,014); S15 tiene una distancia considerable (8,8 bpm) pero mejora ligeramente sin normalizar (+0,007).

**Por qué se mantiene la normalización pese a no tener evidencia estadística concluyente:**
1. La dirección del efecto es consistentemente negativa al quitarla (delta medio y mediano negativos, mayoría de sujetos empeora), aunque no alcance significación.
2. Existe una hipótesis fisiológica razonable — corregir el offset individual de HR basal entre sujetos — parcialmente respaldada por los datos, aunque no aislada de forma limpia (contraejemplos S3, S15 sin explicación identificada).
3. Coherencia metodológica con el diseño original de replicación de Schmidt et al. (2018), que normaliza por defecto.
4. El coste de mantenerla es bajo (no introduce complejidad relevante ni reduce el tamaño muestral de forma significativa: 398 filas normalizadas vs. 376 en bruto, diferencia atribuible a los lags recalculados en esta ablación, no a la normalización en sí).

**Limitación declarada:** n=11 sujetos es una muestra insuficiente para aislar con confianza el efecto de la normalización sobre el rendimiento del clasificador, y para explicar por completo los casos discordantes (S3, S15). Esta ablación debe presentarse como evidencia exploratoria que motiva la decisión de diseño, no como una demostración estadística — una réplica con mayor tamaño muestral sería necesaria para una conclusión firme. Ver también la discusión de compatibilidad delta_hr / HR_mean_norm entre entrenamiento y despliegue (Evolución del proyecto, 6.2/6.4), que sigue siendo un punto abierto independiente de esta ablación.

# 6. Validación empírica: ¿aporta ACC mejora real de rendimiento en Camino A?

Camino A se excluyó de variables de acelerometría (ACC) por delimitación conceptual del proyecto a proxies autonómicos del eje SAM-HPA (HR, HRV, EDA, TEMP), sin verificación empírica previa. Se comprueba aquí si esa exclusión tiene coste real de rendimiento, replicando el mismo patrón de comparación pareada por sujeto ya usado en 3.5.

Se construye una variante de Camino A que incorpora las columnas de ACC (34 columnas frente a las 29 de Camino A original), manteniendo el mismo tratamiento temporal y de normalización, y se evalúa mediante la misma CV anidada de la Sección 3.1 para permitir una comparación directa.

In [31]:
columnas_camino_A_con_ACC = [
    c for c in df_loso.columns
    if c.startswith(('HR_', 'HRV_', 'EDA_', 'TEMP_', 'ACC_'))
    and '_norm' in c and '_3m_' not in c and not c.endswith('_3m')
]

print(f"Camino A + ACC: {len(columnas_camino_A_con_ACC)} columnas (vs. {len(columnas_camino_A)} sin ACC)")

X_A_con_ACC = df_loso[columnas_camino_A_con_ACC]

resultados_nested_A_con_ACC = cv_anidada(X_A_con_ACC, y, groups, rf_base, param_dist_rf, n_iter=30, seed=SEED)
print("\n=== CV anidada — Camino A + ACC (RandomForest) ===")
print(resultados_nested_A_con_ACC[['Sujeto_test', 'Accuracy', 'F1_macro']].sort_values('F1_macro').to_string(index=False))
print(f"\nF1 macro medio: {resultados_nested_A_con_ACC['F1_macro'].mean():.4f} (± {resultados_nested_A_con_ACC['F1_macro'].std():.4f})")

Camino A + ACC: 34 columnas (vs. 29 sin ACC)

=== CV anidada — Camino A + ACC (RandomForest) ===
Sujeto_test  Accuracy  F1_macro
         S3  0.702703  0.412698
         S2  0.714286  0.416667
         S9  0.810811  0.707345
        S14  0.833333  0.824675
        S16  0.833333  0.824675
        S15  0.833333  0.824675
        S17  0.945946  0.935315
         S8  0.972222  0.966387
        S13  0.972973  0.966757
         S4  0.972222  0.968057
         S7  1.000000  1.000000

F1 macro medio: 0.8043 (± 0.2119)


## 6.1. Comparación pareada por sujeto: Camino A con y sin ACC

In [32]:
comparacion_ACC = resultados_nested_A[['Sujeto_test', 'F1_macro']].merge(
    resultados_nested_A_con_ACC[['Sujeto_test', 'F1_macro']],
    on='Sujeto_test', suffixes=('_sin_ACC', '_con_ACC')
)
comparacion_ACC['Delta_F1_macro'] = comparacion_ACC['F1_macro_con_ACC'] - comparacion_ACC['F1_macro_sin_ACC']
print(comparacion_ACC.sort_values('Delta_F1_macro').to_string(index=False))
print(f"\nDelta medio: {comparacion_ACC['Delta_F1_macro'].mean():.4f}")

Sujeto_test  F1_macro_sin_ACC  F1_macro_con_ACC  Delta_F1_macro
         S2          0.732143          0.416667       -0.315476
         S9          0.853175          0.707345       -0.145830
        S15          0.824675          0.824675        0.000000
        S13          0.966757          0.966757        0.000000
        S16          0.824675          0.824675        0.000000
         S3          0.412698          0.412698        0.000000
         S4          0.968057          0.968057        0.000000
        S14          0.824675          0.824675        0.000000
         S7          1.000000          1.000000        0.000000
        S17          0.862963          0.935315        0.072352
         S8          0.851852          0.966387        0.114535

Delta medio: -0.0249


In [33]:
# ======================================================================================================
# TEST DE WILCOXON PAREADO: Camino A con ACC vs. Camino A sin ACC
# ======================================================================================================
stat_ACC, p_ACC = wilcoxon(comparacion_ACC['F1_macro_sin_ACC'], comparacion_ACC['F1_macro_con_ACC'])
print(f"Wilcoxon (pareado, sin ACC vs. con ACC): estadístico={stat_ACC:.3f}, p={p_ACC:.4f}")

Wilcoxon (pareado, sin ACC vs. con ACC): estadístico=3.000, p=0.6250


## 6.2. Conclusión

Añadir ACC a Camino A no mejora el rendimiento agregado (F1 macro medio: 0,8043 con ACC vs. 0,8292 sin ACC; Δ medio = −0,0249), y el efecto es heterogéneo entre sujetos: perjudica notablemente a dos de ellos (S2, S9), no afecta a siete, y beneficia levemente a otros dos (S17, S8). El test de Wilcoxon pareado (n=11, estadístico=3,0, p=0,6250) confirma que esta diferencia no es estadísticamente significativa.

**Decisión:** se mantiene la exclusión de ACC en Camino A, ahora con respaldo empírico y estadístico, y no solo conceptual. La inclusión de ACC no aporta mejora neta, no alcanza significación estadística, y añade heterogeneidad de rendimiento entre sujetos sin beneficio claro que la justifique.

# 7. Entrenamiento del modelo final (Camino B)

Con las decisiones de diseño ya validadas empíricamente (exclusión de ventanas de 3 minutos, normalización por sujeto, exclusión de ACC) y la configuración de hiperparámetros fijada en 3.5 (`n_estimators=100, min_samples_split=20, class_weight='balanced'`), se entrena el modelo final de `StressLoad(t)` sobre la totalidad del conjunto de entrenamiento (398 filas, 11 sujetos), sin partición adicional. El modelo y sus artefactos asociados (dataset `df_loso`, lista de columnas esperadas) se serializan y almacenan en la ruta de artefactos del proyecto, para su uso posterior en la validación n-of-1.

In [34]:
import joblib

modelo_final_camino_B = RandomForestClassifier(
    n_estimators=100, min_samples_split=20, class_weight='balanced', random_state=SEED
)
modelo_final_camino_B.fit(X_B, y)

joblib.dump(modelo_final_camino_B, 'modelo_stress_camino_B_final.pkl')

print(f"Modelo entrenado con {X_B.shape[0]} filas de {groups.nunique()} sujetos.")
print(f"Columnas esperadas como input (en este orden): {list(columnas_camino_B)}")

Modelo entrenado con 398 filas de 11 sujetos.
Columnas esperadas como input (en este orden): ['HR_mean_norm', 'HR_mean_roll_mean_5m_norm', 'HR_mean_roll_std_5m_norm', 'HR_mean_roll_mean_10m_norm', 'HR_mean_roll_std_10m_norm', 'HR_mean_norm_lag_1m_norm', 'HR_mean_norm_lag_2m_norm']


In [35]:
import json
from pathlib import Path
import shutil

with open("WESAD_config_rutas.json") as f:
    config_rutas = json.load(f)

rutas_art = config_rutas["rutas_artefactos_stressload"]
Path(rutas_art["carpeta"]).mkdir(parents=True, exist_ok=True)

# Mover el modelo ya guardado (antes en el directorio de trabajo actual) a su ruta final
shutil.move("modelo_stress_camino_B_final.pkl", rutas_art["modelo_final"])

df_loso.to_parquet(rutas_art["df_loso"], engine="fastparquet", index=False)

with open(rutas_art["columnas_camino_B"], "w") as f:
    json.dump(list(columnas_camino_B), f)

print(f"Artefactos guardados en: {rutas_art['carpeta']}")
print(f"  - modelo: {rutas_art['modelo_final']}")
print(f"  - df_loso: {df_loso.shape[0]} filas, {df_loso['ID_sujeto'].nunique()} sujetos")
print(f"  - columnas_camino_B: {len(columnas_camino_B)} columnas")

Artefactos guardados en: ../models/stressload_camino_B
  - modelo: ../models/stressload_camino_B/modelo_stress_camino_B_final.pkl
  - df_loso: 398 filas, 11 sujetos
  - columnas_camino_B: 7 columnas
